In [ ]:
import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from QuantStudio.Core.QSObject import Panel
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Tools.Visualization import qs_help

# 风险模型检验

风险模型检验用于评估风险模型对组合风险预测的准确度。QuantStudio 的风险模型检验功能位于 `BackTest.Risk` 子模块，采用 Bias Tests 方法，建立在回测基本框架之上。

相关概念见 [基本框架](基本框架.ipynb)，底层调度与扩展接口见 [原理和细节](原理和细节.ipynb)。

## 模块概览

`QuantStudio.BackTest.Risk` 提供以下测试组件：

| 组件 | 算子（计算层） | 回测节点（报告层） | 功能 |
|------|---------------|-------------------|------|
| Bias 检验 | `CalcPortfolioVolatility` / `CalcRandomPortfolio` | `BiasTest` | 检验风险模型对各类投资组合的风险预测准确度 |

该模块遵循"算子 + 回测节点"两层架构，其中 `BiasTest` 节点内部自动生成了多个投资组合并计算每个组合的 Z-Score 和 Bias 统计量。

## 理论基础概述

Bias 检验的核心思想是：将组合已实现收益率 $r_t$ 除以模型预测的风险 $\hat{\sigma}_t$ 得到 Z-Score，若模型预测准确，Z-Score 的标准差应接近 1。在此基础上计算滚动 Bias 统计量（Z-Score 的滚动标准差）和稳健 Bias 统计量（截尾版本），并与 95% 置信区间 $[1-\sqrt{2/T}, 1+\sqrt{2/T}]$ 比较，判断模型是低估还是高估了风险。此外，RAD 统计量（Bias 统计量偏离 1 的绝对均值）用于衡量平均预测偏差。

`BiasTest` 自动生成以下几类投资组合来全面检验风险模型：

1. **全体股票组合**：市值加权和等权的全部 A 股组合
2. **行业组合**：市值加权和等权的各行业组合
3. **行业中性组合**：各行业内因子值排名前一半/后一半的股票组成的等权组合
4. **风格因子组合**：风格因子的 Top 20%/Bottom 20% 组合（5 分位组合）
5. **随机组合**：市值加权和等权的随机抽取 N 只股票的组合（默认 N ∈ {20, 50, 100, 200}）

本篇示例在临时目录中生成确定性的行情与因子数据，无需预先准备数据库。真实使用时将数据来源替换为自己的因子表，并确认日期、证券截面与信息可用时点一致。

## BiasTest — 回测计算节点

`BiasTest` 是继承自 `BTNode` 的回测节点，内部自动生成多种类型的投资组合（全体组合、行业组合、行业中性组合、风格因子 Top/Bottom 组合、随机组合），计算每个组合的 Z-Score、Bias 统计量和稳健 Bias 统计量。

### 构造参数

```python
BiasTest(
    descriptor_ids, price, risk_table, mask=None,
    weight_list=[DataFactor(1, args={"Name": "等权"})],
    style_list=[], industry=None,
    industry_neutral_factor_list=[], extra_portfolio_list=[],
    args={}, config_file=None, **kwargs
)
```

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `descriptor_ids` | `List[str]` | — | 依赖因子的截面 ID 序列 |
| `price` | `Factor` | — | 证券价格/净值因子，用于计算组合收益率 |
| `risk_table` | `RiskTable` | — | 风险表对象，提供协方差矩阵用于计算组合波动率 |
| `mask` | `Optional[Factor]` | `None` | 筛选条件因子，值为 1 的截面才参与计算 |
| `weight_list` | `List[Factor]` | `[等权因子]` | 权重因子列表，默认元素为等权因子 |
| `style_list` | `List[Factor]` | `[]` | 风格因子列表（用于生成风格 Top/Bottom 组合） |
| `industry` | `Optional[Factor]` | `None` | 行业因子，非 None 时自动生成行业组合和行业中性组合 |
| `industry_neutral_factor_list` | `List[Factor]` | `[]` | 行业中性因子列表（在各行业内按因子值取 Top/Bottom 50%） |
| `extra_portfolio_list` | `List[Factor]` | `[]` | 额外自定义投资组合因子列表 |
| `args` | `dict` | `{}` | 参数集 |

### args 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"Bias Test"` | 节点名称（冻结） |
| `IndustryList` | `List[str]` | `[]` | 行业列表（冻结，指定 `industry` 时必填） |
| `RandomNums` | `List[int]` | `[20, 50, 100, 200]` | 随机组合的持仓数量列表（冻结） |
| `RebalanceDTs` | `Optional[List[datetime]]` | `None` | 再平衡时点，None 表示每个时点都再平衡（冻结） |
| `RollingAvgPeriod` | `int` | `12` | Bias 统计量滚动窗口期数（冻结） |

## BiasTestReport — 报告生成节点

`BiasTestReport` 继承自 `ReportNode`，依赖 `BiasTest` 节点，生成包含汇总统计量的 HTML 报告。

```python
from QuantStudio.BackTest.Risk.BiasTest import BiasTestReport

BiasTestReportNode = BiasTestReport(BiasTestNode)
# 或指定报告键名
BiasTestReportNode = BiasTestReport(BiasTestNode, args={"ReportKey": "Bias报告"})
```

### 自动生成的投资组合

`BiasTest` 在构造时对每类投资组合（每个权重因子 × 每个策略）自动生成一个因子：

**全体组合**（每个权重因子 1 个）：
- `"全体{权重}加权组合"`：所有股票的市值/等权组合

**行业组合**（每个权重因子 × 每个行业 1 个，需要指定 `industry` 和 `IndustryList`）：
- `"{行业}行业{权重}加权组合"`：单个行业内股票的市值/等权组合

**行业中性组合**（每个权重因子 × 每个行业中性因子 × Top/Bottom，需要指定 `industry` 和 `industry_neutral_factor_list`）：
- `"{因子}行业中性Top{权重}加权组合"`：各行业内因子值排名前 50% 的股票组合
- `"{因子}行业中性Bottom{权重}加权组合"`：各行业内因子值排名后 50% 的股票组合

**风格因子组合**（每个权重因子 × 每个风格因子 × Top/Bottom，需要指定 `style_list`）：
- `"{风格}风格Top{权重}加权组合"`：因子值排名前 20% 的股票组合
- `"{风格}风格Bottom{权重}加权组合"`：因子值排名后 20% 的股票组合

**随机组合**（每个权重因子 × 每个 `RandomNums` 元素 1 个）：
- `"随机{N}{权重}加权组合"`：随机选取 N 只股票的组合

> **注意**：如果指定了行业因子，`IndustryList` 参数不能为空，且参数中列出的行业名称必须与行业因子中实际存在的行业名称匹配。

### 构造示例

```python
from QuantStudio.BackTest.Risk.BiasTest import BiasTest, BiasTestReport
from QuantStudio.Factor.Factor import DataFactor

# 等权因子（默认就有的权重因子）
EqualWeight = DataFactor(1, args={"Name": "等权"})

BiasTestNode = BiasTest(
    descriptor_ids=SectionIDs,
    price=Price,
    risk_table=RT,
    mask=Mask,
    industry=Industry,
    args={
        "RebalanceDTs": BalanceDTs,
        "IndustryList": ["TMT", "Ind", "Fin"],
        "RandomNums": [5],
        "RollingAvgPeriod": 3,
    }
)

# 创建报告节点
BiasTestReportNode = BiasTestReport(BiasTestNode)
```

### 输出数据结构

`BiasTest.backward_compute` 返回的字典包含：

| 键 | 类型 | 说明 |
|------|------|------|
| `"Z-Score"` | `DataFrame` | 各组合各期的 Z-Score 时序（$z_t = r_t / \hat{\sigma}_t$） |
| `"Robust Z-Score"` | `DataFrame` | 经截尾处理（$[-3, 3]$ 区间）的 Z-Score |
| `"Bias 统计量"` | `DataFrame` | 滚动 Bias 统计量时序，前两列为 95% 置信上下界 |
| `"Robust Bias 统计量"` | `DataFrame` | 滚动稳健 Bias 统计量时序，前两列为 95% 置信上下界 |
| `"汇总统计量"` | `DataFrame` | 各组合的 RAD、高估/低估比例、准确度汇总 |

**汇总统计量指标说明**：
- `RAD 统计量`：$|b_t^T - 1|$ 的均值，越接近 0 表示预测越准确
- `Robust RAD 统计量`：基于稳健 Bias 统计量的 RAD
- `Bias 统计量高估比例`：Bias 统计量低于 95% 置信下界的比例（高估风险）
- `Bias 统计量低估比例`：Bias 统计量高于 95% 置信上界的比例（低估风险）
- `Bias 统计量准确度`：Bias 统计量落在置信区间内的比例
- `Robust *`：对应的稳健版本

## 支持的辅助算子

`BiasTest` 内部使用了几个辅助算子，了解它们有助于深入理解 Bias 检验的计算流程。

### CalcPortfolioVolatility — 组合波动率计算算子

`CalcPortfolioVolatility` 是一个 `SectionOperator`，计算投资组合基于风险模型的波动率预测值 $\hat{\sigma}_t = \sqrt{w^T \mathbf{V} w}$，其中 $\mathbf{V}$ 是风险模型提供的协方差矩阵。

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `*portfolio` | `Factor` | 投资组合因子列表（权重因子） |
| `risk_table` | `RiskTable` | 风险表对象 |
| `portfolio_name_list` | `Optional[List[str]]` | 组合名称列表 |

### CalcPortfolioReturn — 组合收益率计算算子

位于 `BackTest.Strategy.AllocationStrategy`，计算投资组合的已实现收益率 $r_t$。

### CalcRandomPortfolio — 随机组合生成算子

`CalcRandomPortfolio` 是一个 `SectionOperator`，生成随机选取 N 只股票的投资组合。

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `weight` | `Factor` | 权重因子（如等权或市值加权） |
| `mask` | `Optional[Factor]` | 筛选条件因子 |

### Bias 检验的计算流程

`BiasTest` 节点在构造时内部执行的步骤：

1. **生成投资组合**：根据 `weight_list`、`industry`、`style_list`、`industry_neutral_factor_list`、`RandomNums` 等参数，通过 `_genPortfolio` 方法生成大量投资组合因子
2. **计算组合收益率**：通过 `CalcPortfolioReturn` 计算每个组合各期的已实现收益率 $r_t$
3. **计算组合波动率**：通过 `CalcPortfolioVolatility` 从风险表中读取协方差矩阵 $\mathbf{V}$，计算每个组合各期的预测波动率 $\hat{\sigma}_t = \sqrt{w^T \mathbf{V} w}$
4. **计算 Z-Score**：$z_t = r_t / \text{Lag}(\hat{\sigma}_t)$，其中波动率取滞后一期（上月预测 × 当月已实现）
5. **汇总分析**：在 `backward_compute` 中计算滚动 Bias 统计量、稳健 Bias 统计量、RAD 统计量等

## 完整示例：已有组合的 Bias 检验

下面在本篇生成行情和风险数据，构造固定等权与行业组合，计算实际日收益和前一日预测波动率之比，再通过 `BiasTest.template().bind(zscore=...).build()` 汇总统计并生成报告。使用该入口时，Z-Score 的列名就是 `PortfolioNameList`，上下文截面使用组合名称，底层证券顺序在权重矩阵中显式对齐。

`BiasTest(...)` 的自动组合构造接口仍列于前面的 API 说明；当前源码的该路径存在截面和回溯范围衔接问题，本篇不将它作为已验证的完整示例。已有组合的检验可以使用下面的公开依赖入口。

In [ ]:
# 在临时目录中创建独立的演示库，所有数据由固定随机种子生成。
demo_directory = TemporaryDirectory()
demo_root = Path(demo_directory.name)
(demo_root / "Cache").mkdir()
factor_dir = demo_root / "HDF5"
factor_dir.mkdir()
FDB = HDF5DB(args={"MainDir": factor_dir}).connect()
rng = np.random.default_rng(0)
demo_dates = list(pd.date_range("2025-01-01", "2025-04-30").to_pydatetime())
demo_ids = [f"{i:06d}.SZ" for i in range(1, 16)]
demo_shape = (len(demo_dates), len(demo_ids))
demo_price = pd.DataFrame(
    10 * np.exp(np.cumsum(rng.normal(.001, .01, demo_shape), axis=0)),
    index=demo_dates, columns=demo_ids,
)
FDB.writeData(Panel({"close": demo_price, "open": demo_price * .999,
                     "high": demo_price * 1.01, "low": demo_price * .99}),
              table_name="stock_cn_day_bar")
FDB.writeData(Panel({"if_listed": pd.DataFrame(1., index=demo_dates, columns=demo_ids)}),
              table_name="stock_cn_status")
FDB.writeData(Panel({"industry": pd.DataFrame(
    np.tile(["TMT", "Ind", "Fin"] * 5, (len(demo_dates), 1)),
    index=demo_dates, columns=demo_ids)}),
    table_name="stock_cn_industry", data_type={"industry": "string"})
FDB.writeData(Panel({name: pd.DataFrame(rng.uniform(.01, .2, demo_shape),
                                      index=demo_dates, columns=demo_ids)
                    for name in ("bp_lr", "ep_ttm")}), table_name="stock_cn_factor_value")
FDB.writeData(Panel({"close": demo_price.mean(axis=1).to_frame("000905.SH")}),
              table_name="index_cn_day_bar")

from QuantStudio.Risk.HDF5RDB import HDF5FRDB

risk_dir = demo_root / "Risk"
risk_dir.mkdir()
RDB = HDF5FRDB(args={"MainDir": risk_dir}).connect()
risk_factors = ["市场", "风格"]
exposure = pd.DataFrame({"市场": np.ones(len(demo_ids)),
                         "风格": np.linspace(-1, 1, len(demo_ids))}, index=demo_ids)
factor_cov = pd.DataFrame(np.diag([.0001, .0002]), index=risk_factors, columns=risk_factors)
specific_risk = pd.Series(.01, index=demo_ids)
for date in demo_dates:
    RDB.writeData("demo_risk_table", date, factor_data=exposure, factor_cov=factor_cov,
                  specific_risk=specific_risk,
                  factor_ret=pd.Series([.001, -.001], index=risk_factors),
                  specific_ret=pd.Series(0., index=demo_ids))

In [ ]:
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext
from QuantStudio.Factor.Factor import DataFactor, FactorContext
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.BackTest.Risk.BiasTest import BiasTest, BiasTestReport

# 固定等权及行业组合，每日再平衡；风险预测使用前一日期的数据。
SectionIDs = demo_ids
RT = RDB.getTable("demo_risk_table")
price_data = FDB.getTable("stock_cn_day_bar").readData(
    factor_names=["close"], ids=SectionIDs, dts=demo_dates).loc["close"]
weights = pd.DataFrame({
    "全体等权": np.full(len(SectionIDs), 1 / len(SectionIDs)),
    "TMT等权": np.tile([1., 0., 0.], 5) / 5,
}, index=SectionIDs)
portfolio_returns = price_data.pct_change().iloc[1:] @ weights
prior_covariance = RT.readCov(dts=demo_dates[:-1], ids=SectionIDs)
predicted_volatility = pd.DataFrame(
    [np.sqrt(np.diag(weights.T @ prior_covariance.loc[date] @ weights))
     for date in demo_dates[:-1]],
    index=demo_dates[1:], columns=weights.columns,
)
z_scores = portfolio_returns / predicted_volatility

# 已有 Z-Score 时，可通过公开具名依赖构造统计节点。
portfolio_names = list(weights.columns)
BiasTestNode = BiasTest.template(args={
    "PortfolioNameList": tuple(portfolio_names), "RollingAvgPeriod": 20,
}).bind(zscore=DataFactor(z_scores)).build()
BiasReportNode = BiasTestReport(BiasTestNode)
Report = BTReport(report_nodes=[BiasReportNode])
with FactorContext(DTRuler=demo_dates, SectionIDs=portfolio_names) as context:
    Rslt = Engine().run([Report], context,
                        fwd_data_list=[DTLocalContext(DTs=demo_dates[1:])])
stats = Rslt[0]["0-" + BiasReportNode.Name]
np.testing.assert_allclose(stats["Z-Score"], z_scores)
print(stats["汇总统计量"])

In [ ]:
from IPython.display import HTML, display

display(HTML(Rslt[0]["Report"]))

## 输出解读

`Rslt[0]["Report"]` 为汇总 HTML；`Rslt[0]["0-" + BiasReportNode.Name]` 为风险检验的结构化结果，包含 `Z-Score`、`Bias 统计量`、`Robust Bias 统计量` 和 `汇总统计量`。

Bias 接近 1 表示预测风险与实现风险的尺度相近。短样本仅用于验证 API，不能用于评价风险模型质量；实际评估需要足够长的样本和一致的收益、协方差频率。

窗口长度以 Z-Score 的行数计；这里使用 20 个日样本。价格收益与风险预测均为日频，避免将月收益直接除以日波动率。